In [ ]:
# Run ID from config/runs.py (1-25), or a list such as [7, 12] for two runs on two GPUs.
RUN_ID = 7
# True: tiny subset and one epoch, to check the pipeline. False: the full run.
SMOKE = False

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/CoDraft-Bench.git"
BRANCH = "new"
DATA_ROOT = "/kaggle/input/datasets/nhtquyn/codraft"


def sh(*cmd: str, **kw: object) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, **kw)


if os.path.isdir("/kaggle/working"):
    OUT_ROOT = "/kaggle/working"
    REPO_DIR = f"{OUT_ROOT}/CoDraft-Bench"
    if os.path.isdir(REPO_DIR):
        sh("git", "-C", REPO_DIR, "pull", "--ff-only")
    else:
        sh("git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR)
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt")
else:
    REPO_DIR = os.getcwd()
    while REPO_DIR != "/" and not os.path.isfile(os.path.join(REPO_DIR, "runner.py")):
        REPO_DIR = os.path.dirname(REPO_DIR)
    OUT_ROOT, DATA_ROOT = REPO_DIR, None

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
commit = subprocess.run(
    ["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True
).stdout.strip()
branch = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True
).stdout.strip()
print(f"Repo {REPO_DIR} | branch {branch} | commit {commit}")
assert branch == BRANCH, f"Expected branch {BRANCH}, got {branch}"

In [ ]:
from config.runs import describe
from runner import find_data_root, launch

print(describe(), "\n")
launch(RUN_ID, data_root=find_data_root(DATA_ROOT), out_root=OUT_ROOT, smoke=SMOKE)

In [ ]:
import glob
import json

from IPython.display import Image, display

ids = [RUN_ID] if isinstance(RUN_ID, int) else RUN_ID
for f in sorted(glob.glob(f"{OUT_ROOT}/results/*_metrics.json")):
    with open(f) as fh:
        r = json.load(fh)
    if r["run_id"] not in ids or r["smoke"] != SMOKE:
        continue
    print(f"\n{r['tag']}  [{r['table']}] {r['row']}")
    print(
        f"  Macro-F1 {r['f1_macro'] * 100:.2f} | QWK {r['qwk']:.4f} | MAE {r['mae']:.4f} | "
        f"adjacent {r['adjacent_acc'] * 100:.2f}% | severe {r['severe_rate'] * 100:.2f}% | n={r['n_test']}"
    )
    print(
        "  Per-class F1: " + " / ".join(f"{c['name']} {c['f1'] * 100:.2f}" for c in r["per_class"])
    )
    print(f"  {r['train_seconds'] / 60:.1f} min on {r['environment']['gpu']}")
    png = f"{OUT_ROOT}/results/cm_{r['tag']}.png"
    if os.path.isfile(png):
        display(Image(png, width=480))

zips = sorted(glob.glob(f"{OUT_ROOT}/*codraft_runs_*.zip"), key=os.path.getmtime)
if zips:
    print(f"\nDownload this file: {zips[-1]} ({os.path.getsize(zips[-1]) / 2**20:.1f} MB)")
else:
    print("\nNo zip found; the launch cell may have failed before packaging.")